# Final Revision 06: Cross-Phase Statistical and Consistency Audit

Run this notebook **after Phases 0-5 have been rerun with the final-revision notebooks**.

It creates a single traceable results registry, deduplicates repeated cross-phase comparisons, applies Holm-Bonferroni correction across prespecified inferential families, and checks that significance-table MAEs agree with the final ensemble metrics. Phase 5 is labeled exploratory and kept separate from the main Phase 0-4 inferential families.


In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive/Crypto_Research')
ROOT = PROJECT_DIR / 'revised_outputs_v3'
OUT = ROOT / 'final_revision_audit'
OUT.mkdir(parents=True, exist_ok=True)
PHASE_DIRS = {
    'phase0': ROOT / 'phase0',
    'phase1': ROOT / 'phase1',
    'phase2': ROOT / 'phase2',
    'phase3': ROOT / 'phase3',
    'phase4': ROOT / 'phase4',
    'phase5': ROOT / 'phase5_reliability_fusion',
}


In [ ]:

# FINAL REVISION: one deterministic implementation of Holm-Bonferroni for the cross-phase family audit.
def holm_adjust(p_values):
    values = np.asarray(p_values, dtype=float)
    out = np.full(values.shape, np.nan, dtype=float)
    idx = np.where(np.isfinite(values))[0]
    if len(idx) == 0:
        return out
    p = values[idx]
    order = np.argsort(p)
    p_sorted = p[order]
    m = len(p_sorted)
    adj_sorted = np.empty(m)
    running = 0.0
    for rank, value in enumerate(p_sorted):
        running = max(running, min(1.0, (m-rank) * float(value)))
        adj_sorted[rank] = running
    unsorted = np.empty(m)
    unsorted[order] = adj_sorted
    out[idx] = unsorted
    return out

metric_frames = []
sig_frames = []
for phase, directory in PHASE_DIRS.items():
    metric_path = directory / 'ensemble_metrics.csv'
    sig_path = directory / 'significance_final.csv'
    if metric_path.exists():
        frame = pd.read_csv(metric_path)
        frame.insert(0, 'source_phase', phase)
        frame['source_file'] = str(metric_path)
        metric_frames.append(frame)
    if sig_path.exists():
        frame = pd.read_csv(sig_path)
        frame.insert(0, 'source_phase', phase)
        frame['source_file'] = str(sig_path)
        sig_frames.append(frame)

if not metric_frames or not sig_frames:
    raise FileNotFoundError('Final phase outputs are incomplete. Rerun Phases 0-5 before the consistency audit.')

metrics = pd.concat(metric_frames, ignore_index=True, sort=False)
significance = pd.concat(sig_frames, ignore_index=True, sort=False)


In [ ]:

# FINAL REVISION: build a unique metric registry so manuscript numbers are copied from one source of truth.
metric_cols = ['source_phase','asset','phase','model','method','mae','rmse','r2','directional_accuracy','directional_accuracy_p_value','observation_count','source_file']
metric_cols = [c for c in metric_cols if c in metrics.columns]
registry = metrics[metric_cols].copy()
registry['result_id'] = (
    registry['source_phase'].astype(str) + '__' + registry['asset'].astype(str) + '__' + registry['method'].astype(str)
)
registry = registry.sort_values(['source_phase','asset','method']).drop_duplicates('result_id', keep='last')
if registry['result_id'].duplicated().any():
    raise AssertionError('Duplicate result IDs remain in the final registry.')
registry.to_csv(OUT / 'paper1_results_registry.csv', index=False)
display(registry)


In [ ]:
# FINAL REVISION: repeated method outputs across phase notebooks must agree numerically.
repeat_rows = []
for (asset, method), group in metrics.groupby(['asset', 'method']):
    if len(group) <= 1:
        continue
    for metric in ['mae', 'rmse', 'r2', 'directional_accuracy']:
        if metric not in group.columns:
            continue
        values = pd.to_numeric(group[metric], errors='coerce').dropna()
        if len(values) == 0:
            continue
        repeat_rows.append({
            'asset': asset, 'method': method, 'metric': metric,
            'occurrences': int(len(values)), 'min_value': float(values.min()),
            'max_value': float(values.max()), 'range': float(values.max()-values.min()),
            'pass_1e_10': bool((values.max()-values.min()) <= 1e-10),
        })
repeated_method_consistency = pd.DataFrame(repeat_rows)
repeated_method_consistency.to_csv(OUT / 'repeated_method_consistency_check.csv', index=False)
display(repeated_method_consistency)
if not repeated_method_consistency.empty and not repeated_method_consistency['pass_1e_10'].all():
    raise AssertionError('Repeated method metrics differ across phase notebooks after canonical reuse. Inspect repeated_method_consistency_check.csv before manuscript drafting.')


# V3 CONSISTENCY FIX: verify that aliased references are exact copies of their canonical prediction artifacts.
CANONICAL_ALIASES = [
    ('phase0', 'phase0', 'phase1', 'phase0'),
    ('phase0', 'phase0', 'phase2', 'phase0'),
    ('phase0', 'phase0', 'phase3', 'phase0'),
    ('phase2', 'phase2', 'phase3', 'phase2_general'),
    ('phase2', 'phase2', 'phase5', 'phase2_general'),
    ('phase3', 'phase3_crypto', 'phase4', 'phase3_reference'),
    ('phase3', 'phase3_crypto', 'phase5', 'phase3_crypto'),
]

def _compare_prediction_alias(source_file_name, source_phase_dir, source_phase_key, target_phase_dir, target_phase_key):
    source_path = PHASE_DIRS[source_phase_dir] / source_file_name
    target_path = PHASE_DIRS[target_phase_dir] / source_file_name
    rows = []
    if not source_path.exists() or not target_path.exists():
        return rows
    source = pd.read_csv(source_path)
    target = pd.read_csv(target_path)
    for model in ['lstm','xgb']:
        sm = f'{source_phase_key}_{model}'
        tm = f'{target_phase_key}_{model}'
        left = source[(source['phase'].astype(str)==source_phase_key) & (source['method'].astype(str)==sm)].copy()
        right = target[(target['phase'].astype(str)==target_phase_key) & (target['method'].astype(str)==tm)].copy()
        keys = ['asset','seed','target_date']
        if 'walk_forward_fold' in left.columns and 'walk_forward_fold' in right.columns:
            keys.append('walk_forward_fold')
        cols = keys + ['actual_return','predicted_return']
        left = left[cols].rename(columns={'actual_return':'actual_source','predicted_return':'pred_source'})
        right = right[cols].rename(columns={'actual_return':'actual_target','predicted_return':'pred_target'})
        merged = left.merge(right, on=keys, how='outer', indicator=True)
        complete = merged['_merge'].eq('both')
        actual_diff = np.abs(merged.loc[complete,'actual_source'] - merged.loc[complete,'actual_target'])
        pred_diff = np.abs(merged.loc[complete,'pred_source'] - merged.loc[complete,'pred_target'])
        passed = bool(
            len(merged) > 0
            and complete.all()
            and (actual_diff <= 1e-12).all()
            and (pred_diff <= 1e-12).all()
        )
        rows.append({
            'file': source_file_name,
            'source_phase_dir': source_phase_dir,
            'source_method': sm,
            'target_phase_dir': target_phase_dir,
            'target_method': tm,
            'rows_compared': int(complete.sum()),
            'max_actual_abs_diff': float(actual_diff.max()) if len(actual_diff) else np.nan,
            'max_prediction_abs_diff': float(pred_diff.max()) if len(pred_diff) else np.nan,
            'pass_1e_12': passed,
        })
    return rows

alias_rows=[]
for src_dir, src_key, tgt_dir, tgt_key in CANONICAL_ALIASES:
    alias_rows.extend(_compare_prediction_alias('predictions.csv', src_dir, src_key, tgt_dir, tgt_key))
    alias_rows.extend(_compare_prediction_alias('walk_forward_predictions.csv', src_dir, src_key, tgt_dir, tgt_key))
canonical_reuse = pd.DataFrame(alias_rows)
canonical_reuse.to_csv(OUT / 'canonical_prediction_reuse_check.csv', index=False)
display(canonical_reuse)
if canonical_reuse.empty or not canonical_reuse['pass_1e_12'].all():
    raise AssertionError('Canonical prediction reuse check failed. Do not draft the manuscript until all alias checks pass.')


In [ ]:

# FINAL REVISION: remove comparisons repeated in later notebooks, then correct multiplicity across each scientific family.
key = ['asset','candidate_method','reference_method']
sig = significance.sort_values('source_phase').drop_duplicates(key, keep='first').reset_index(drop=True)

# V3 CONSISTENCY FIX: each p-value is linked to the exact phase-native metric rows used by that comparison.
sig['candidate_result_id'] = sig['source_phase'].astype(str) + '__' + sig['asset'].astype(str) + '__' + sig['candidate_method'].astype(str)
sig['reference_result_id'] = sig['source_phase'].astype(str) + '__' + sig['asset'].astype(str) + '__' + sig['reference_method'].astype(str)
registry_ids = set(registry['result_id'].astype(str))
missing_ids = sorted((set(sig['candidate_result_id']) | set(sig['reference_result_id'])) - registry_ids)
if missing_ids:
    raise AssertionError(f'Significance registry references missing result IDs: {missing_ids[:10]}')


# Phase 5 is exploratory by design and is never pooled into a Phase 0-4 primary family.
sig['final_family'] = sig['comparison_family'].astype(str)
sig.loc[sig['source_phase'].eq('phase5'), 'final_family'] = 'phase5_exploratory'

sig['dm_p_value_holm_cross_phase_family'] = np.nan
sig['wilcoxon_p_value_holm_cross_phase_family'] = np.nan
for family, idx in sig.groupby('final_family').groups.items():
    idx = list(idx)
    sig.loc[idx, 'dm_p_value_holm_cross_phase_family'] = holm_adjust(sig.loc[idx, 'dm_p_value_two_sided'])
    sig.loc[idx, 'wilcoxon_p_value_holm_cross_phase_family'] = holm_adjust(sig.loc[idx, 'wilcoxon_p_value_two_sided'])

sig.to_csv(OUT / 'paper1_significance_registry.csv', index=False)
display(sig.sort_values(['final_family','asset','comparison']))


In [ ]:

# V3 CONSISTENCY FIX: numerical traceability uses exact source-phase result IDs, never cross-phase method averages.
metric_lookup = registry.set_index('result_id')['mae'].to_dict()
checks = []
for _, row in sig.iterrows():
    candidate_registry = metric_lookup.get(row['candidate_result_id'], np.nan)
    reference_registry = metric_lookup.get(row['reference_result_id'], np.nan)
    checks.append({
        'source_phase': row['source_phase'],
        'asset': row['asset'], 'comparison': row['comparison'],
        'candidate_method': row['candidate_method'], 'reference_method': row['reference_method'],
        'candidate_result_id': row['candidate_result_id'], 'reference_result_id': row['reference_result_id'],
        'candidate_mae_significance': row['candidate_mae'], 'candidate_mae_registry': candidate_registry,
        'reference_mae_significance': row['reference_mae'], 'reference_mae_registry': reference_registry,
        'candidate_abs_diff': abs(row['candidate_mae']-candidate_registry) if np.isfinite(candidate_registry) else np.nan,
        'reference_abs_diff': abs(row['reference_mae']-reference_registry) if np.isfinite(reference_registry) else np.nan,
    })
consistency = pd.DataFrame(checks)
consistency['pass_1e_10'] = (
    consistency['candidate_abs_diff'].notna()
    & consistency['reference_abs_diff'].notna()
    & (consistency['candidate_abs_diff'] <= 1e-10)
    & (consistency['reference_abs_diff'] <= 1e-10)
)
consistency.to_csv(OUT / 'metric_significance_consistency_check.csv', index=False)
display(consistency)
if not consistency['pass_1e_10'].all():
    raise AssertionError('Significance-to-registry traceability failed. Inspect metric_significance_consistency_check.csv.')
print(f'PASS: {int(consistency["pass_1e_10"].sum())}/{len(consistency)} significance-to-registry checks.')


In [ ]:

# Collect revision-specific evidence files without inventing results when a phase has not yet been rerun.
evidence_files = {
    'temporal_alignment': OUT / 'temporal_alignment_protocol.json',
    'timestamp_audit': OUT / 'timestamp_convention_audit.csv',
    'phase4_decay_selection': PHASE_DIRS['phase4'] / 'decay_rate_selection_summary.csv',
    'phase4_decay_lock': PHASE_DIRS['phase4'] / 'selected_decay_rate.json',
    'phase5_reliability_sensitivity': PHASE_DIRS['phase5'] / 'reliability_coverage_sensitivity.csv',
    'phase5_provider_shift': PHASE_DIRS['phase5'] / 'provider_shift_weight_diagnostics.csv',
}
evidence_status = pd.DataFrame([
    {'item': k, 'path': str(v), 'exists': v.exists()} for k,v in evidence_files.items()
])
evidence_status.to_csv(OUT / 'final_revision_evidence_status.csv', index=False)
display(evidence_status)

checklist = """# Paper 1 final consistency checklist

- Require `canonical_prediction_reuse_check.csv`, `repeated_method_consistency_check.csv`, and `metric_significance_consistency_check.csv` to pass before manuscript drafting.
- Copy all reported MAE/RMSE/R2/DA values from `paper1_results_registry.csv`.
- Copy inferential p-values from `paper1_significance_registry.csv`; identify raw vs Holm-adjusted values explicitly.
- Use DM-HAC as the primary forecast-loss comparison and two-sided Wilcoxon as the nonparametric robustness check.
- Do not compare classification accuracy values from prior literature numerically with this study's regression MAE/RMSE/R2.
- State the UTC news window, forecast origin, and close-to-close target interval explicitly.
- Report the validation-selected Phase 4 decay half-life/rate exactly as stored in `selected_decay_rate.json`.
- Use five-seed walk-forward summaries, not the legacy seed-42-only result.
- Keep Phase 5 exploratory and brief; report the provider-volume sensitivity audit.
- Regenerate final figures from final CSV outputs; do not transcribe intermediate console numbers.
"""
(OUT / 'manuscript_consistency_checklist.md').write_text(checklist, encoding='utf-8')
print(checklist)
